# Lesson 6.4 — Health, Events, Tiers, and Metric Names Query Types

## Learning Objectives
- Use the Health query type to visualize AppDynamics health rule violations
- Use the Events query type to query application events and configure annotations
- Use the Tiers query type to get tier information
- Use the Metric Names query type to browse the metric hierarchy

**Reference**: [Health, Events, Tiers, and Metric Names Query Types](https://grafana.com/docs/plugins/dlopes7-appdynamics-datasource/latest/query-editor/#health-events-tiers-and-metric-names-query-types) | [Annotations](https://grafana.com/docs/plugins/dlopes7-appdynamics-datasource/latest/annotations/)

## 1. Query Types Overview

All four remaining query types use **form-based editors** (not free-text query entry) and query the **AppDynamics Controller REST API** (not the Events Service).

| Query Type | What it returns | Primary Use |
|------------|-----------------|-------------|
| **Health** | Health rule violations for an application | Show app health status in a panel |
| **Events** | Application events (deployments, errors, policy violations) | Annotations on time series panels |
| **Tiers** | Tier information (name, node count, agent type) | Populate template variable dropdowns |
| **Metric Names** | Browse the metric hierarchy for an application | Explore available metrics |

### Common controls:
All four support a **"Use dashboard's time picker"** toggle:
- **ON** (default): query automatically uses the dashboard's global time range
- **OFF**: specify a fixed start and end time in the query fields

## 2. Health Query Type

Shows health rule violation status for an application.

### Configuration:
1. Query type: **Health**
2. **Application**: Select your application
3. **Use dashboard's time picker**: Toggle ON/OFF
4. If OFF: set start-time and end-time manually

### Output:
Returns health rule violations with:
- Violation start/end time
- Health rule name
- Severity (WARNING/CRITICAL)
- Affected entity name
- Resolution status

### Best panel type: **State Timeline** or **Table**
- State Timeline: visualizes health state changes over time per entity
- Table: shows a list of all violations in the time range

## 3. Events Query Type and Annotations

The Events query type queries AppDynamics application events — particularly useful for **annotations** on time-series panels.

### What are Grafana Annotations?
Annotations appear as **vertical lines or markers** on time series panels. They mark significant events: deployments, config changes, alerts, incidents.

When you add an AppDynamics annotation to a dashboard, deployment events and health rule violations appear as lines on your response time and error rate charts — making root cause correlation visual and immediate.

### Setting up annotations:
1. In your Grafana dashboard, go to **Settings → Annotations**
2. Click **+ Add annotation query**
3. **Data source**: AppDynamics
4. **Query type**: Events
5. **Application**: StoreFront
6. **Event types**: Select: Application Deployment, Error, Policy Violation
7. Save — deployment and policy events will now appear as vertical lines on all panels

**Reference**: [AppDynamics Annotations](https://grafana.com/docs/plugins/dlopes7-appdynamics-datasource/latest/annotations/)

In [ ]:
# Demonstrate the annotation concept
import matplotlib.pyplot as plt
import random

random.seed(5)
minutes = list(range(120))

# Baseline response time with an incident after a deployment
rt = [random.gauss(300, 40) for _ in range(60)]  # normal before deployment
rt += [random.gauss(1200, 200) for _ in range(20)]  # incident after deployment
rt += [random.gauss(320, 45) for _ in range(40)]  # recovery after fix

# Simulated AppDynamics events
events = [
    {"time": 60, "type": "APPLICATION_DEPLOYMENT",   "label": "Deploy v2.1.0", "color": "blue"},
    {"time": 63, "type": "HEALTH_RULE_VIOLATION",   "label": "RT Critical",   "color": "red"},
    {"time": 80, "type": "POLICY_VIOLATION_RESOLVED", "label": "RT Resolved",  "color": "green"},
    {"time": 82, "type": "APPLICATION_DEPLOYMENT",   "label": "Rollback v2.0", "color": "orange"},
]

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(minutes, rt, color='steelblue', linewidth=1.5, label='Avg Response Time')
ax.fill_between(minutes, rt, alpha=0.1, color='steelblue')

# Add annotation vertical lines
for event in events:
    ax.axvline(x=event['time'], color=event['color'], linestyle='--', linewidth=1.5, alpha=0.8)
    ax.text(event['time'] + 0.5, max(rt) * 0.92, event['label'],
            fontsize=7, color=event['color'], rotation=90, va='top')

ax.axhline(300 * 3, color='orange', linestyle=':', linewidth=1, label='Slow threshold')
ax.set_xlabel('Minute')
ax.set_ylabel('Avg Response Time (ms)')
ax.set_title('Response Time with AppDynamics Event Annotations in Grafana')
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('annotations.png', dpi=100, bbox_inches='tight')
plt.show()
print("Annotations show deployment events causing and resolving the incident.")
print("This makes root cause correlation immediate and visual.")

## 4. Tiers and Metric Names Query Types

### Tiers query type:
Returns the list of tiers for an application.

**Primary use**: Populate **template variable** dropdowns.

To create a `${tier}` variable that lists all tiers dynamically:
1. Dashboard Settings → Variables → Add variable
2. Type: **Query**
3. Data source: AppDynamics
4. Query type: **Tiers**
5. Application: `${application}` (or hardcoded name)

Now the tier dropdown auto-populates from the live AppDynamics API.

### Metric Names query type:
Returns the metric name hierarchy for browsing.

**Primary use**: Explore available metrics when building queries.

This is the equivalent of the Metric Browser in the AppDynamics Controller UI — you can navigate the hierarchy to find the exact metric path for a Metrics query.

In [ ]:
# Summary: All six query types in one reference

query_types_reference = [
    {
        "type": "Metrics",
        "api": "Controller REST API → /metric-data",
        "auth": "Controller username/password",
        "returns": "Time-series metric values",
        "panel_types": ["Time Series", "Stat", "Gauge", "Bar Chart"],
        "covered_in": "Lesson 6.2"
    },
    {
        "type": "Analytics",
        "api": "Analytics Events API → /events/query",
        "auth": "Analytics API Key + Global Account Name",
        "returns": "Event records and aggregations (ADQL)",
        "panel_types": ["Time Series", "Table", "Stat", "Pie", "Bar"],
        "covered_in": "Lesson 6.3"
    },
    {
        "type": "Health",
        "api": "Controller REST API → /healthrule-violations",
        "auth": "Controller username/password",
        "returns": "Health rule violation records",
        "panel_types": ["State Timeline", "Table"],
        "covered_in": "Lesson 6.4 (this lesson)"
    },
    {
        "type": "Events",
        "api": "Controller REST API → /events",
        "auth": "Controller username/password",
        "returns": "Application events (deployments, errors, policy violations)",
        "panel_types": ["Annotations", "Table", "Logs"],
        "covered_in": "Lesson 6.4 (this lesson)"
    },
    {
        "type": "Tiers",
        "api": "Controller REST API → /tiers",
        "auth": "Controller username/password",
        "returns": "Tier names and info for an application",
        "panel_types": ["Template variable source"],
        "covered_in": "Lesson 6.4 (this lesson)"
    },
    {
        "type": "Metric Names",
        "api": "Controller REST API → /metrics",
        "auth": "Controller username/password",
        "returns": "Metric hierarchy nodes",
        "panel_types": ["Template variable source", "Exploration tool"],
        "covered_in": "Lesson 6.4 (this lesson)"
    },
]

print("=== AppDynamics Grafana Plugin — All Query Types ===")
print()
for qt in query_types_reference:
    print(f"[{qt['type']:15}] → API: {qt['api']}")
    print(f"  Auth       : {qt['auth']}")
    print(f"  Returns    : {qt['returns']}")
    print(f"  Panel types: {', '.join(qt['panel_types'])}")
    print()

In [ ]:
# EXERCISE 6.4: Design a dashboard that uses multiple query types
#
# TASK: For each panel in the 'Full-Stack Health Dashboard' below,
# specify which query type to use.

dashboard_panels = [
    {
        "panel": "Response Time Trend (last 2 hours)",
        "query_type": "???",   # Metrics / Analytics / Health / Events
        "reason": "???"
    },
    {
        "panel": "Deployment and Incident Annotations on the Response Time chart",
        "query_type": "???",
        "reason": "???"
    },
    {
        "panel": "Health Rule Violation Status Timeline",
        "query_type": "???",
        "reason": "???"
    },
    {
        "panel": "Top 10 Slowest Transactions (ADQL)",
        "query_type": "???",
        "reason": "???"
    },
    {
        "panel": "Tier dropdown template variable (for dashboard filter)",
        "query_type": "???",
        "reason": "???"
    },
]

print("Full-Stack Health Dashboard Panel Analysis:")
for p in dashboard_panels:
    print(f"\n  Panel: {p['panel']}")
    print(f"    Query type: {p['query_type']}")
    print(f"    Reason    : {p['reason']}")

## Summary

- **Health** query: Returns health rule violations → use State Timeline or Table panels
- **Events** query: Returns application events → use as **Annotations** on time-series panels to correlate deployments with performance changes
- **Tiers** query: Returns tier list → use as template variable data source for dynamic tier dropdowns
- **Metric Names** query: Browse metric hierarchy → use for exploration and building Metrics queries
- All four use Controller REST API auth (not Analytics API auth)

---

**Next**: [Lesson 6.5 — Template Variables and Advanced Use Cases](lesson-6.5-template-variables-advanced.ipynb)